# Test 4 — Manual (LLM) Coref at Sentence-Level Chunks

**Motivation (from Tests 1–3):** With `fastcoref` LingMess, coref-before-embed was **flat** on public
benchmarks (Test 2 common-noun corpus, Test 3 entity-rich Wikipedia). An open question remained: is
the ceiling in the *technique* or in the *coref model's quality*? Test 4 removes the model from the
loop entirely and uses **manual (LLM-quality) coreference** — the highest-quality resolution possible —
to establish an upper bound.

**Second claim under test:** *"small chunks lose context."* We chunk at the **sentence level** (one
sentence per chunk), where a bare pronoun ("He resigned in 1971") carries zero entity signal. If
resolving references makes these tiny chunks self-contained and retrievable, coref can substitute for
a larger chunk window as a context-preservation strategy.

**Data:** the three files from `test-4-data/` (Apollo 11 Wikipedia article, ~5k words):

| File | Role |
|------|------|
| `original_chunks.json` | 100 sentence chunks, pronouns intact (baseline) |
| `coref_chunks.json` | Same 100 chunks, manual coref (pronouns → named entities) |
| `eval_questions.json` | 30 questions, gold chunk ids, `coref_critical` flag |

## Variants

| Label | Retrieval |
|-------|-----------|
| **baseline** | Dense on **original** sentence text |
| **coref_dense** | Dense on the **manual coref** rewrite |
| **coref_hybrid** | **Weighted RRF**: 0.7 * dense(coref) + 0.3 * BM25(original) |

**Invariants:** original text is always returned to the user; gold = `gold_chunk_ids`; same chunk ids
across variants (1:1 alignment asserted). Models are identical to Test 3 (`bge-small-en-v1.5` +
`rank_bm25`).

## 1. Setup & Install

Lighter than Test 3: **no `fastcoref`, no `spacy`** — coref is already done (manually) in the input
JSON. We only need the embedding + lexical + metrics stack.

In [ ]:
!pip install -q pytrec_eval tqdm pandas numpy rank_bm25 sentence-transformers

## 2. Config

`DATA_DIR` resolves the Kaggle input path first, then falls back to a local `test-4-data/` folder so
the notebook runs both on Kaggle and locally. On Kaggle, upload the three JSON files as a Dataset
named `test-4-data` (path becomes `/kaggle/input/test-4-data/`).

In [ ]:
import os, logging, glob
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("coref_v4")

# --- Models (identical to Test 3) ---
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
BATCH_SIZE  = 64
MAX_LEN     = 512

# --- Retrieval / fusion knobs (identical to Test 3) ---
TOP_K        = 5
RRF_K        = 60
DENSE_WEIGHT = 0.7           # weighted-RRF weight for dense(coref); BM25(original) gets (1 - this)
RUN_DEPTH    = 100
CACHE_DIR    = "./embed_cache_v4"

# --- Resolve the data folder: Kaggle input first, then local fallback ---
def _resolve_data_dir():
    candidates = [
        "/kaggle/input/test-4-data",
        "/kaggle/input/test4-data",
        "./test-4-data",
        "test-4-data",
    ]
    for c in candidates:
        if os.path.isdir(c) and glob.glob(os.path.join(c, "*chunks*.json")):
            return c
    # last resort: any kaggle input subdir containing our files
    for c in glob.glob("/kaggle/input/*"):
        if glob.glob(os.path.join(c, "*chunks*.json")):
            return c
    raise FileNotFoundError("Could not find test-4-data. Upload the 3 JSON files as a Kaggle Dataset "
                            "or place them in ./test-4-data/")

DATA_DIR = _resolve_data_dir()
os.makedirs(CACHE_DIR, exist_ok=True)
log.info("Config | model=%s | data_dir=%s | dense_w=%.2f | top_k=%d", EMBED_MODEL, DATA_DIR, DENSE_WEIGHT, TOP_K)

## 3. Load the three JSON files (1:1 chunk alignment)

Chunk ids are the shared key across `original_chunks.json` and `coref_chunks.json`. Questions carry
`gold_chunk_ids` and a `coref_critical` flag (True when the gold chunk's *original* text uses only a
pronoun for the entity the question names — the case coref should help).

In [ ]:
import json

def _load(name):
    with open(os.path.join(DATA_DIR, name), "r", encoding="utf-8") as f:
        return json.load(f)

def load_test4():
    original = _load("original_chunks.json")
    coref    = _load("coref_chunks.json")
    questions = _load("eval_questions.json")

    orig_by_id  = {c["chunk_id"]: c["text"] for c in original}
    coref_by_id = {c["chunk_id"]: c["text"] for c in coref}
    assert set(orig_by_id) == set(coref_by_id), "chunk id mismatch between original and coref"

    chunk_ids = sorted(orig_by_id)                       # deterministic order
    passages = [{"_id": str(cid),
                 "text": orig_by_id[cid],
                 "coref_text": coref_by_id[cid]} for cid in chunk_ids]

    queries, qrels, critical = {}, {}, {}
    for q in questions:
        qid = str(q["q_id"])
        queries[qid] = q["question"]
        qrels[qid] = {str(g): 1 for g in q["gold_chunk_ids"]}
        critical[qid] = bool(q.get("coref_critical", False))

    log.info("Loaded %d chunks | %d questions (%d coref-critical) | data_dir=%s",
             len(passages), len(queries), sum(critical.values()), DATA_DIR)
    return {"name": "apollo11_sentences", "passages": passages,
            "queries": queries, "qrels": qrels, "critical": critical}

ds = load_test4()
# quick peek: a coref-critical example (original pronoun vs manual rewrite)
_ex = ds["passages"][3]   # chunk_id 4: "Aldrin joined him ..."
print("ORIG :", _ex["text"])
print("COREF:", _ex["coref_text"])

## 4. Encoder (bge-small dense) with disk cache (same as Test 3)

In [ ]:
import hashlib, pickle, re
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

_has_cuda = False
try:
    import torch
    _has_cuda = torch.cuda.is_available()
except Exception:
    pass

PRONOUN_RE = re.compile(r"\b(he|she|him|her|his|hers|they|them|their|theirs|it|its)\b", re.IGNORECASE)

def _hash_texts(texts):
    h = hashlib.sha1()
    for t in texts:
        h.update(t.encode("utf-8", "ignore")); h.update(b"\x00")
    return h.hexdigest()[:16]

def _cache_file(dataset, variant, role, texts):
    return os.path.join(CACHE_DIR, f"{dataset}__{variant}__{role}__{_hash_texts(texts)}.pkl")

dense_model = SentenceTransformer(EMBED_MODEL, device="cuda" if _has_cuda else "cpu")
log.info("Loaded %s (dense) + rank_bm25 (lexical) | device=%s", EMBED_MODEL, "cuda" if _has_cuda else "cpu")

def encode_cached(dataset, variant, role, texts):
    cf = _cache_file(dataset, variant, role, texts)
    if os.path.exists(cf):
        with open(cf, "rb") as f:
            arr = pickle.load(f)
        log.info("cache HIT  | %s", os.path.basename(cf))
        return arr
    log.info("cache MISS | encoding %d %s texts (%s/%s)", len(texts), role, dataset, variant)
    arr = dense_model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True,
                             show_progress_bar=False).astype("float32")
    with open(cf, "wb") as f:
        pickle.dump(arr, f)
    return arr

## 5. Retrieval: dense, BM25, and weighted RRF (identical to Test 3)

In [ ]:
def dense_ranklist(q_dense, p_dense, depth=RUN_DEPTH):
    sims = q_dense @ p_dense.T
    order = np.argsort(-sims, axis=1)[:, :depth]
    return [[(int(j), float(sims[qi, j])) for j in order[qi]] for qi in range(sims.shape[0])]

def bm25_ranklist(query_texts, passage_texts, depth=RUN_DEPTH):
    from rank_bm25 import BM25Okapi
    tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    bm25 = BM25Okapi([tok(t) for t in passage_texts])
    out = []
    for q in tqdm(query_texts, desc="BM25", mininterval=2.0):
        scores = bm25.get_scores(tok(q))
        order = np.argsort(-scores)[:depth]
        out.append([(int(j), float(scores[j])) for j in order])
    return out

def weighted_rrf(dense_rl, bm25_rl, dense_w=DENSE_WEIGHT, k=RRF_K, depth=RUN_DEPTH):
    """Weighted reciprocal-rank fusion of dense(coref) and BM25(original)."""
    nq = len(dense_rl)
    w = {"dense": dense_w, "bm25": 1.0 - dense_w}
    fused = []
    for qi in range(nq):
        acc = {}
        for sysname, rl in (("dense", dense_rl), ("bm25", bm25_rl)):
            for rank, (pidx, _s) in enumerate(rl[qi]):
                acc[pidx] = acc.get(pidx, 0.0) + w[sysname] * 1.0 / (k + rank + 1)
        ordered = sorted(acc.items(), key=lambda x: -x[1])[:depth]
        fused.append([(int(p), float(s)) for p, s in ordered])
    return fused

## 6. Metrics (pytrec_eval) + coref-critical breakdown

Same Recall@5 / nDCG@10 / MRR as Test 3, plus a per-subset recall so we can see whether coref helps
specifically on the **coref-critical** questions (the whole point of the test).

In [ ]:
import pytrec_eval

def build_run(ranklist, query_ids, passage_ids):
    run = {}
    for qi, qid in enumerate(query_ids):
        run[qid] = {passage_ids[pidx]: float(score) for pidx, score in ranklist[qi]}
    return run

def eval_run(run, qrels, critical=None, k=TOP_K):
    ev = pytrec_eval.RelevanceEvaluator(qrels, {"recall_5", "ndcg_cut_10", "recip_rank"})
    per = ev.evaluate(run)
    recall = float(np.mean([v["recall_5"] for v in per.values()])) if per else 0.0
    ndcg = float(np.mean([v["ndcg_cut_10"] for v in per.values()])) if per else 0.0
    mrr = float(np.mean([v["recip_rank"] for v in per.values()])) if per else 0.0
    gold = {q: {d for d, s in rels.items() if s > 0} for q, rels in qrels.items()}
    hits = set()
    for qid, docs in run.items():
        topk = [d for d, _ in sorted(docs.items(), key=lambda x: -x[1])[:k]]
        if gold.get(qid) and (set(topk) & gold[qid]):
            hits.add(qid)
    out = {"recall_at_5": recall, "ndcg_at_10": ndcg, "mrr": mrr, "hits": hits, "n": len(qrels)}
    # subset recall@5 on coref-critical vs non-critical questions
    if critical is not None:
        crit_ids = [q for q in qrels if critical.get(q)]
        non_ids  = [q for q in qrels if not critical.get(q)]
        out["recall_crit"] = float(np.mean([per[q]["recall_5"] for q in crit_ids])) if crit_ids else 0.0
        out["recall_noncrit"] = float(np.mean([per[q]["recall_5"] for q in non_ids])) if non_ids else 0.0
        out["n_crit"] = len(crit_ids); out["n_noncrit"] = len(non_ids)
    return out

## 7. Run the three variants

`coref_hybrid` fuses dense(coref) with BM25 over the **original** text (0.7 / 0.3), matching Test 3.

In [ ]:
def run_dataset(ds):
    name = ds["name"]
    passages = ds["passages"]
    p_ids = [p["_id"] for p in passages]
    q_ids = list(ds["queries"].keys())
    q_texts = [ds["queries"][q] for q in q_ids]
    orig_texts = [p["text"] for p in passages]
    coref_texts = [p["coref_text"] for p in passages]

    q_dense       = encode_cached(name, "query", "query", q_texts)
    p_dense_orig  = encode_cached(name, "baseline", "passage", orig_texts)
    p_dense_coref = encode_cached(name, "coref_dense", "passage", coref_texts)

    rl_baseline = dense_ranklist(q_dense, p_dense_orig)
    rl_corefden = dense_ranklist(q_dense, p_dense_coref)
    rl_bm25     = bm25_ranklist(q_texts, orig_texts)
    rl_hybrid   = weighted_rrf(rl_corefden, rl_bm25)         # 0.7*dense(coref) + 0.3*BM25(orig)

    variants = {"baseline": rl_baseline, "coref_dense": rl_corefden, "coref_hybrid": rl_hybrid}
    results, runs = {}, {}
    for label, rl in variants.items():
        run = build_run(rl, q_ids, p_ids)
        runs[label] = run
        results[label] = eval_run(run, ds["qrels"], ds["critical"])
        r = results[label]
        log.info("[%s/%s] R@5=%.3f nDCG@10=%.3f MRR=%.3f | R@5(crit)=%.3f R@5(non)=%.3f",
                 name, label, r["recall_at_5"], r["ndcg_at_10"], r["mrr"],
                 r["recall_crit"], r["recall_noncrit"])
    return {"name": name, "results": results, "runs": runs,
            "n_passages": len(passages), "n_queries": len(q_ids)}

## 8. Results table + flip analysis

In [ ]:
import pandas as pd

def summarize(run_out):
    res = run_out["results"]; base = res["baseline"]
    rows = []
    for label in ["baseline", "coref_dense", "coref_hybrid"]:
        r = res[label]
        rows.append({"variant": label, "recall@5": round(r["recall_at_5"],4),
                     "nDCG@10": round(r["ndcg_at_10"],4), "MRR": round(r["mrr"],4),
                     "R@5_crit": round(r["recall_crit"],4), "R@5_noncrit": round(r["recall_noncrit"],4),
                     "Δrecall@5": round(r["recall_at_5"]-base["recall_at_5"],4),
                     "Δcrit": round(r["recall_crit"]-base["recall_crit"],4)})
    df = pd.DataFrame(rows)
    print(f"\n=== {run_out['name']} | passages={run_out['n_passages']} queries={run_out['n_queries']} ===")
    print(df.to_string(index=False))
    return df

def flips(run_out):
    res = run_out["results"]; base_hits = res["baseline"]["hits"]
    all_qids = set(run_out["runs"]["baseline"].keys()); out = {}
    for label in ["coref_dense", "coref_hybrid"]:
        h = res[label]["hits"]
        rec, hurt, bf = h - base_hits, base_hits - h, all_qids - base_hits - h
        out[label] = {"recovered": sorted(rec), "hurt": sorted(hurt), "both_fail": sorted(bf)}
        print(f"\n[{run_out['name']}] {label} vs baseline: recovered={len(rec)} hurt={len(hurt)} both_fail={len(bf)}")
        if rec:  print("   recovered q_ids:", sorted(rec, key=int))
        if hurt: print("   hurt q_ids     :", sorted(hurt, key=int))
    return out

run_out = run_dataset(ds)
df = summarize(run_out)
fl = flips(run_out)

## 9. Worked example — a sentence chunk recovered by coref

Pick a query where `coref_dense` recovered a baseline miss and show the gold chunk's original
(pronoun-only) text vs the manual coref rewrite that injected the named entity.

In [ ]:
def show_worked_example(ds, run_out, variant="coref_dense"):
    res = run_out["results"]
    recovered = res[variant]["hits"] - res["baseline"]["hits"]
    by_id = {p["_id"]: p for p in ds["passages"]}
    if not recovered:
        print(f"No recovery for {variant}. Showing a coref-critical gold chunk for context instead.")
        qid = next((q for q in ds["queries"] if ds["critical"].get(q)), None)
        if qid is None: return
        recovered = {qid}
    for qid in sorted(recovered, key=int):
        gold_ids = {pid for pid, s in ds["qrels"][qid].items() if s > 0}
        for gid in sorted(gold_ids, key=int):
            p = by_id.get(gid)
            if p and p["coref_text"] != p["text"]:
                print("QUERY:", ds["queries"][qid], "| coref_critical=", ds["critical"].get(qid))
                print("\n--- Gold chunk", gid, f"({variant} vs baseline) ---")
                print("RETURNED (original):", p["text"])
                print("EMBEDDED (manual coref):", p["coref_text"])
                print("pronouns  orig:", len(PRONOUN_RE.findall(p["text"])),
                      "| coref:", len(PRONOUN_RE.findall(p["coref_text"])))
                return
    print("Recovered queries exist but none had a changed gold chunk.")

show_worked_example(ds, run_out)

## 10. Write `test-4-findings.md` (results section)

Appends the measured results to the existing `test-4-findings.md` design doc (or writes it fresh if
run standalone). Falls back to the notebook directory if the repo root isn't writable.

In [ ]:
def _md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

def write_results(path="test-4-findings.md"):
    import time
    ro = run_out
    out = []
    out.append("\n\n---\n")
    out.append("## Measured Results (auto-generated)\n")
    out.append(f"**Run date:** {time.strftime('%Y-%m-%d')} | **Notebook:** `coref_public_eval_v4.ipynb`  ")
    out.append(f"**Models:** {EMBED_MODEL} (dense) + rank_bm25 (lexical) | "
               f"weighted-RRF {DENSE_WEIGHT:.1f}/{1-DENSE_WEIGHT:.1f} | TOP_K={TOP_K}\n")
    out.append(f"Chunks: {ro['n_passages']} (sentence-level) | Queries: {ro['n_queries']} "
               f"({ro['results']['baseline']['n_crit']} coref-critical, "
               f"{ro['results']['baseline']['n_noncrit']} non-critical)\n")
    out.append(_md_table(df) + "\n")
    base_hits = ro["results"]["baseline"]["hits"]; all_qids = set(ro["runs"]["baseline"].keys())
    out.append("**Flips vs baseline**\n")
    out.append("| variant | recovered | hurt | both_fail |")
    out.append("| --- | --- | --- | --- |")
    for label in ["coref_dense", "coref_hybrid"]:
        h = ro["results"][label]["hits"]
        out.append(f"| {label} | {len(h-base_hits)} | {len(base_hits-h)} | {len(all_qids-base_hits-h)} |")
    out.append("\n**How to read this:** if `R@5_crit` rises for `coref_dense` while `R@5_noncrit` stays "
               "flat, manual coref helps exactly where it should (pronoun-only gold chunks). A flat "
               "`R@5_crit` would mean even perfect coref can't beat the dense model's implicit context "
               "handling at sentence level.\n")
    text = "\n".join(out) + "\n"

    for target in [path, os.path.join(os.getcwd(), os.path.basename(path))]:
        try:
            mode = "a" if os.path.exists(target) else "w"
            with open(target, mode, encoding="utf-8") as f:
                f.write(text)
            log.info("Wrote results to %s (mode=%s, %d chars)", target, mode, len(text))
            break
        except Exception as e:
            log.warning("could not write %s: %s", target, e)
    print(text)

write_results()